In [ ]:
import sqlite3
from dotenv import load_dotenv
import os
import psycopg

load_dotenv()

Extract data from SQLite db

In [ ]:
SQLITEconn = sqlite3.connect("fiftyville.db")
SQLITEcur = SQLITEconn.cursor()

SQLITEcur.execute("SELECT * FROM sqlite_master")

schema = sorted(SQLITEcur.fetchall(), 
    key=lambda x: x[3]
)

Due to datatype discrepancies between SQLite and PostgreSQL, the loop approach doesn't work and I had to create each table manually. 

In [ ]:
# for table in schema:
#     print("CREATE QUERY", table[4])
#     SUPABASEcur.execute(table[4])
#     SUPABASEconn.commit()
#     query = f"SELECT * FROM {table[1]}"
#     SQLITEcur.execute(query)
#     result = SQLITEcur.fetchall()
#     params = ""
#     for i, v in enumerate(result[0]):
#         params += "%s"
#         if i != len(result[0]) - 1:
#             params += ","
#     query = f"INSERT INTO {table[1]} VALUES ({params})"
    
#     print("INSERT QUERY", query)
#     print("INSERTING", result)
    
#     SUPABASEcur.executemany(query, result)
#     SUPABASEconn.commit()

In [ ]:
for table in schema:
    print(table[1], "=>", table[4])

In [ ]:
table = schema[9][1]

query = f"SELECT * FROM {table}"
SQLITEcur.execute(query)
result = SQLITEcur.fetchall()
params = ""
for i, v in enumerate(result[0]):
    params += "%s"
    if i != len(result[0]) - 1:
        params += ","
query = f"INSERT INTO {table} VALUES ({params})"

print("INSERT QUERY", query)
print("INSERTING", result)

In [ ]:
dbconn = os.getenv("DBCONN")
SUPABASEconn = psycopg.connect(dbconn)
SUPABASEcur = SUPABASEconn.cursor()

SUPABASEcur.executemany(query, result)
SUPABASEconn.commit()

SUPABASEcur.close()
SUPABASEconn.close()

In [ ]:
SQLITEcur.close()
SQLITEconn.close()

## Re-format Data

The format for the tables is designed for learning, but for this project I have found some of the data shapes to be difficult to work with in this context. I am consolidating all dates and times into timestamp strings. 

In [ ]:
from supabase import create_client

supabase = create_client(
        os.getenv("SUPABASE_URL"), 
        os.getenv("SUPABASE_KEY"))

In [ ]:
current_table = "phone_calls"

In [ ]:
# data = supabase.table(current_table).select().is_("date", "null").execute().data
data = supabase.table(current_table).select().execute().data

In [ ]:
data

In [ ]:
# dates
upsert_data = []

for row in data:
  month = f"0{row["month"]}" if len(str(row["month"])) == 1 else row["month"]
  day = f"0{row["day"]}" if len(str(row["day"])) == 1 else row["day"]
  upsert_data.append({ "id": row["id"], "date": f"{row["year"]}-{month}-{day}" })

In [ ]:
# times
upsert_data = []

for row in data:
  hour = f"0{row["hour"]}" if len(str(row["hour"])) == 1 else row["hour"]
  minute = f"0{row["minute"]}" if len(str(row["minute"])) == 1 else row["minute"]
  upsert_data.append({ "id": row["id"], "time": f"{hour}:{minute}" })

In [ ]:
supabase.table(current_table).upsert(upsert_data).execute()